# Sistema de Recomendación
Este es el notebook creado por Roberto en clase. 
Yo hice una copia en mi Google Drive para correr en Colab pero tarda mucho. Lo ejecuto local. 
La idea es rellenar con código para entrenar el modelo y hacer predicciones. 

In [2]:
import sqlite3
import pandas as pd

In [18]:
# Esta celda no es necesaria si corro local
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# DATABASE = '/content/drive/MyDrive/Colab Notebooks/SR/datos/data.db'
# DATABASE = '/content/data.db'
DATABASE = 'datos/data.db'

# Plan General
Dado que vamos a usar 20 interacciones de cada usuario para testing, necesitamos que cada usuario a incluir en el dataset de training y testing tenga al menos 20 interacciones el que tenga exactamente 20 cuenta como un arranque en frio.
Lo que haremos es formar un dataset de interacciones con todas las interacciones de cada usuario menos 20 tomadas al azar. Entonces, entrenaré el modelo para que pueda predecir, para cada usuario los 20 libros que aparecerían en sus 20 mayores preferencias y los ordenaremos. La función de costo será el NDCG tomada sobre las 20 interacciones seleccionadas para testing.

# Training y testing

In [15]:
# Trae una lista de todos los id_lector que tienen al menos 20 interacciones

con = sqlite3.connect(DATABASE)

id_lectores = pd.read_sql("""
    SELECT id_lector
      FROM interacciones
    GROUP BY id_lector
    HAVING COUNT(*) >= 20
""", con)["id_lector"]

con.close()

In [16]:
# Armamos el dataset de train y test para el dataframe id_lectores
con = sqlite3.connect(DATABASE)

df_train = pd.DataFrame({"id_lector": [], "id_libro": [], "rating": []})
df_test = pd.DataFrame({"id_lector": [], "id_libro": [], "rating": []})

for id_lector in id_lectores:
    id_libros = pd.read_sql("""
        SELECT id_lector, id_libro, rating
          FROM interacciones
         WHERE id_lector = ?
         ORDER BY fecha
    """, con, params=[id_lector])

    # las últimas 20 interacciones son para testing
    df_train = pd.concat([df_train, id_libros[:-20]], axis=0)
    # el resto es para training
    df_test = pd.concat([df_test, id_libros[-20:]], axis=0)

con.close()

In [8]:
df_train.to_csv("datos/train.csv", index=False)
df_test.to_csv("datos/test.csv", index=False)

In [18]:
# Arrancar desde acá a partir de la segunda ejecución
import sqlite3
import pandas as pd
# DATABASE = '/content/data.db'
DATABASE = 'datos/data.db'

df_train = pd.read_csv("datos/train.csv")
df_test = pd.read_csv("datos/test.csv")

In [19]:
df_train.head()

,id_lector,id_libro,rating
0,05-03-1970,niebla,7.0
1,05-03-1970,una-soledad-demasiado-ruidosa,7.0
2,05-03-1970,hadji-murat,8.0
3,05-03-1970,la-muerte-de-ivan-ilich,9.0
4,05-03-1970,quien-domina-el-mundo,8.0


# Recomendador

In [20]:
con = sqlite3.connect(DATABASE)
todos_los_libros = pd.read_sql("SELECT DISTINCT id_libro, genero, autor FROM libros", con)
con.close()


In [21]:
todos_los_libros.head(5)

,id_libro,genero,autor
0,-el-gran-libro-de-los-cuandos,NaN,NaN
1,-incluso-el-olvido,Lecturas complementarias,"VITAS, ROMÁN ROLANDO"
2,007-licencia-para-matar,NaN,NaN
3,1-000-sitios-que-ver-antes-de-morir,NaN,NaN
4,1-000-sitios-que-ver-antes-de-morir-america,NaN,NaN


In [24]:


def retrieval(id_lector):
    """Retorna todos los libros que se pueden recomendar a id_lector"""

    
    libros_leidos = df_train[df_train["id_lector"] == id_lector]["id_libro"].to_list()
    libros_no_leidos = todos_los_libros[~todos_los_libros["id_libro"].isin(libros_leidos)]

    # ToDo: Disminuir la cantidad de libros elegibles para ser recomendados
    # Idea 01: Traer libros de los generos que el usuario calificó y de los autores que calificó (unión de los dos)
    # Idea 02: Enriquecer con géneros y autores asociados a los preferidos por el usuario (aquellos géneros
    # y autores que fueron elegidos conjuntamente con los calificados por el lector)

    generos_preferidos = todos_los_libros[todos_los_libros["id_libro"].isin(libros_leidos)]["genero"].to_list()
    autores_preferidos = todos_los_libros[todos_los_libros["id_libro"].isin(libros_leidos)]["autor"].to_list()
    libros_no_leidos = libros_no_leidos[libros_no_leidos["genero"].isin(generos_preferidos) | libros_no_leidos["autor"].isin(autores_preferidos)]

    # return libros_no_leidos["id_libro"].to_list()
    return libros_no_leidos


In [26]:
# Ejemplo: cuantos libros quedan sin leer para zymu
# Inicialmente, sin filtrar, teníamos 128,720
# Filtrando para incluir solo libros de los autores y generos presentes en training
# tenemos 32,338 es decir un 25% - Está bien para empezar, después puede refinarse
# print(len(retrieval("zymu")))
libros_no_leidos = retrieval("zymu")
libros_no_leidos.head(20)

,id_libro,genero,autor
1,-incluso-el-olvido,Lecturas complementarias,"VITAS, ROMÁN ROLANDO"
13,10-000-anos-mirando-estrellas,Ensayo,"BALLESTEROS, FERNANDO J. y LUQUE, BARTOLOMÉ"
29,10-gritos-contra-la-gordofobia,Ensayo,"PIÑEYRO, MAGDALENA"
30,10-ideas-clave-animacion-a-la-lectura,Lecturas complementarias,"MATA, JUAN"
60,100-clasicos-del-cine-del-siglo-xx,Lecturas complementarias,VV.AA.
85,100-mitos-de-la-ciencia,"Fantástica, ciencia ficción","CLOSA AUTET, DANIEL"
86,100-mitos-de-la-historia-de-mexico-1,No Ficción,"MARTÍN MORENO, FRANCISCO"
95,100-personajes-que-hunden-espana,Lecturas complementarias,"VALENZUELA, CURRI"
118,1000-experiencias-unicas,Lecturas complementarias,VV.AA.
129,1000-sitios-que-ver-antes-de-morir,Lecturas complementarias,"SCHULTZ, PATRICIA"


In [ ]:
# Crear los datasets de train y test
# ToDo: Completar df_train y df_test con los datos que salen de las tablas sin feature engineering
# ToDo Later: Agregar Data Engineering
def construir_dataset(df: pandas.DataFrame):
    pass
    return df

In [ ]:
# Entrenar el Modelo
# ToDo: Definir función de costo tomando el ndcg por lector
# ToDo: Crear rutina de entrenamiento
# ToDo: Optimizar HP con Bayesiana de Optuna
# ToDo: Entrenar y guardar modelo entrenado


In [ ]:
# Obtener predicciones
# ToDo: Crear lista de ids de usuarios para recomendar
# ToDo: Filtrar libros a recomendar para cada uno
# ToDo: Generar predicciones 
# ToDo: Guardar archivo
import random
def ranking(id_lector, id_libros):
    """Predice el rating de los libros en id_libros para el lector id_lector y los devuelve los mejores N"""

    return dict([(id_libro, random.random()) for id_libro in id_libros])


In [ ]:
id_libros = retrieval("zymu")
ranking("zymu", id_libros)

In [ ]:
from sklearn.metrics import ndcg_score
import numpy as np

df_test = pd.read_csv("test.csv")

ndcg_lista = []

for id_lector in df_test["id_lector"].unique():
    libros_a_recomendar = retrieval(id_lector)

    true_relevance_dict = df_test.loc[df_test["id_lector"] == id_lector, ["id_libro", "rating"]].set_index("id_libro").to_dict()["rating"]
    predicted_scores_dict = ranking(id_lector, libros_a_recomendar)


    id_libros = list(set(list(true_relevance_dict.keys()) + list(predicted_scores_dict.keys())))

    y_true = np.asarray(
        [[true_relevance_dict.get(id_libro, 0) for id_libro in id_libros]]
    )
    y_score = np.asarray(
        [[predicted_scores_dict.get(id_libro, 0) for id_libro in id_libros]]
    )

    ndcg_lista.append(ndcg_score(y_true, y_score, k=20))

    print(f"{ndcg_score(y_true, y_score, k=20):0.5f}")

print(np.mean(ndcg_lista))


0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000
0.00000


KeyboardInterrupt: 